In [1]:
!pip install -q groq gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 2.1 MB/s eta 0:00:00


器材借用系統(v.mini)

In [2]:
#library
import sqlite3
from google.colab import userdata #important
from groq import Groq
import gradio as gr

In [3]:
#api
client = Groq(api_key=userdata.get('SDDA_Lecture05'))
print ('Successfully access Groq API')

Successfully access Groq API


In [4]:
#database
conn = sqlite3.connect('equipment.db')
cursor = conn.cursor()
print ('Successfully connect to database')

Successfully connect to database


In [5]:
#database:prepare
cursor.execute('DROP TABLE IF EXISTS items')
cursor.execute('DROP TABLE IF EXISTS borrow_logs')

In [6]:
#database : table buliding
cursor.execute("""
  CREATE TABLE items(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT,
    status TEXT)
""")

In [7]:
#database:table building- borrow_logs
cursor.execute("""
CREATE TABLE borrow_logs(
id INTEGER PRIMARY KEY AUTOINCREMENT,
item_name TEXT,
borrow_time TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")

In [8]:
#database : insert data for testing

initial_data = [('單眼相機','可借用'),('腳架','以借出'),('投影機','可借用')]
cursor.executemany('INSERT INTO items (name,status) VALUES(?,?)',initial_data)
conn.commit()
conn.close()

print('Successfully INSERT item into table (items)')

Successfully INSERT item into table (items)


In [9]:
def ai_find_item(user_say):
  prompt = f"""
  系辦目前有三種設備:[單眼相機、腳架、投影機]。
  請從學生的話裡面找出他想藉用哪種器材。
  規則:只能回傳器材名稱(例如:單眼相機、腳架、投影機)。如果學生說的不是這三種，請回傳「未知」。絕對不要給其他多餘的文字!

  學生的話: {user_say}
  """
  response = client.chat.completions.create(
      model = 'openai/gpt-oss-120b',
      messages = [{'role':'user','content':prompt}],
      temperature = 0.0
  )
  item_name = response.choices[0].message.content.strip()
  return item_name

In [10]:
#just test
result = ai_find_item('系秘好，我下午要拍片，想要一台相機。')
print(result)

單眼相機


In [11]:
user_say = '系秘好，我下午要拍片，想要一台相機。'

In [16]:
#database: checking
def borrow_system(user_say):
  #1. check what student want to borrow by AI
  target = ai_find_item(user_say)
  print(f'try to find {target} in item ...')

  if target not in ['單眼相機、腳架、投影機']:
    return f'I cannot find {target} in item database.'

  #2. connect to db to check the item is available
  conn = sqlite3.connect('equipment.db')
  cursor = conn.cursor()

  cursor.execute('SELECT status FROM items WHERE name = ?', (target,))
  row = cursor.fetchone()
  current_status = row[0]
  print(f'[RESULT]{target} status is {current_status} !')

  #3. check the item is available
  if current_status == '以借出':
    conn.close()
    return f'Sorry, the {target} is not available.'
  else :
  #4.執行借出程序
    cursor.execute('UPDATE items SET status = "以借出" WHERE name = ?',(target,))
    cursor.execute('INSERT INTO borrow_logs (item_name) VALUES (?)',(target,))
    conn.commit()
    conn.close()
    return f'Successfully borrow {target},and change status into 以借出'

In [17]:
print (borrow_system('我想借用單眼相機'))

try to find 單眼相機 in item ...
I cannot find 單眼相機 in item database.


In [19]:
def web_interface(message):
  if not message.stripo():
    return '請輸入想借用什麼器材!'
  reply = borrow_system(message)
  return reply

demo = gr.Interface(
    fn = web_interface,
    inputs=gr.Textbox(label = '輸入您的需求'),
    outputs = gr.Textbox(label = '系統回復'),
    title = '簡易器材借用系統',
    description = '結合Groq與SQLite的奇才借用系統',
)
demo.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9c2219efa83c084b8e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
